# Jude 2026a · touch typing T17 and T18

## Prepare the data

Run the commands below from the repository root with `bci-repro` active. See the [reader guide](../../docs/20261124_repository_migration/reader_workflow.md) for environment setup.

### 1. Download

Download `Data.zip` from [Dryad](https://doi.org/10.5061/dryad.cz8w9gjjk). Create `studies/20260316_jude/data/raw/dryad_cz8w9gjjk/` and save the download as `studies/20260316_jude/data/raw/dryad_cz8w9gjjk/dryad_cz8w9gjjk__Data.zip`.

### 2. Extract

Extract the archive into `studies/20260316_jude/data/extracted/dryad_cz8w9gjjk/`.

```text
python -m zipfile -e "studies/20260316_jude/data/raw/dryad_cz8w9gjjk/dryad_cz8w9gjjk__Data.zip" "studies/20260316_jude/data/extracted/dryad_cz8w9gjjk"
```

After extraction, check for `studies/20260316_jude/data/extracted/dryad_cz8w9gjjk/Data/t17/`. The extracted `Data/` directory also contains `t18/`.

### 3. Prepare the model input

Convert the session MAT files to 20 ms TFRecords, then export an NPZ for PyTorch. This example uses T17's April 25 session, blocks 1, 3, 4, and 5, with 20 test trials.

```text
python shared/tools/setup_upstream.py --study 20260316_jude
python studies/20260316_jude/code/prepare_data.py t17.2024.04.25 1,3,4,5 20
python studies/20260316_jude/code/pytorch_port/export_tfrecord_to_npz.py t17.2024.04.25
```

**TFRecord output** `studies/20260316_jude/data/derived/dryad_cz8w9gjjk/tfrecord/typing_20ms/t17/t17.2024.04.25/`.

**NPZ output** `studies/20260316_jude/data/derived/dryad_cz8w9gjjk/npz/typing_20ms/t17/t17.2024.04.25/m1_torch_data.npz`.

### 4. Export the common session file

Write one typing session to the session format shared by every study in this repository. The file stores every cue of every block with the released 10 ms features and the publisher's timing variables, and records the block file and the cue the trial came from. This example exports T17's April 25, 2024 session, the day step 3 converts.

```text
python studies/20260316_jude/code/export_sessions.py --session t17.2024.04.25
```

**Output** `studies/20260316_jude/data/processed/t17/dryad_cz8w9gjjk_2024-04-25.h5`. Add `--list` to see every session and whether it is exported, or use `--all` to export them all.

Check the file structure and trace every trial back to its publisher file with one command.

```text
python shared/tools/validate_session.py studies/20260316_jude/data/processed/t17/dryad_cz8w9gjjk_2024-04-25.h5 --source trials
```

**Output** A `PASS` line for the file, ending with `source trace: 80/80 trials re-read and identical`.

In [1]:
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "figures/reproduction_benchmark.yaml").is_file())
sys.path.insert(0, str(root / "shared"))
from session_format import read_session

session_file = root / "studies/20260316_jude/data/processed/t17/dryad_cz8w9gjjk_2024-04-25.h5"
if session_file.is_file():
    with read_session(session_file) as session:
        print(f"{session.attrs['session']}: {len(session)} trials")
        trial = session.trials[0]
        print(trial.trial_id, "·", trial.target("sentence"))
        for name in trial.names("neural"):
            array = trial.neural(name)
            print(f"  neural/{name}: {array.shape} {array.dtype}, "
                  f"{trial.group[f'neural/{name}'].attrs['bin_ms']} ms bins")
        chars = session.label_sets["chars"]
        print("  characters:", "".join(chars[i] for i in trial.target("char_class_ids")[:12]), "...")
else:
    print("Session file not found. Run the export command above to create it.")

t17.2024.04.25: 80 trials
t17.2024.04.25/block01/0000 · Is it going outside?
  neural/spike_pow: (8759, 384) float32, 10 ms bins
  neural/tx: (8759, 384) int16, 10 ms bins
  characters: IS>IT>GOING> ...


## Read the archived results

The released test trial split was used for checkpoint validation. Figure 3b applies a minimum of three trials per session. Raw greedy CER is available at session level. The per-trial character and word counts below describe the post-LM output.

This walkthrough uses the checked-in archive and takes no data download.

**Which trials are scored, and how to compare with the paper.** The paper's evaluation split is the one the release ships; this pipeline reuses it, so a fresh `trials.csv` covers the same trials as the archived score. **"Same test set" means same sessions *and* same trials, every trial in the same order — matching recording dates alone is not enough.** Where to verify: check your fresh `trials.csv` against [`data/splits/dryad_cz8w9gjjk__archived_evaluation.csv`](data/splits/dryad_cz8w9gjjk__archived_evaluation.csv) row for row (session and reference text, in the same order), or run `python studies/20260316_jude/code/build_split_manifest.py`, which fails unless the exported evaluation trials equal those archived rows in content and order. The split columns are documented in [`data/splits/README.md`](data/splits/README.md) and the paper-figure selection in the [evaluation protocol](docs/20261124_evaluation_protocol/README.md). Once the trial check passes, match the score-bearing run length before quoting a fresh WER against the paper — the full recipe is [Section 5 of the fresh-machine guide](../../fresh_machine_setup.md).

In [2]:
import csv
import json
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "figures/reproduction_benchmark.yaml").is_file())
sys.path.insert(0, str(root / "figures/code"))
from figure_paths import PUBLISHED, load_benchmark

benchmark = load_benchmark()

def read_rows(path):
    """Read the archived CSV rows in their original order."""
    with (root / path).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

## From a trial or day to the reported result

In [3]:
from build_session_cer_vs_wer import build_jude26a
source = benchmark["figure3_cer_sources"]["Jude 2026a (T17)"]
archive = json.loads((root / source["archive"]).read_text(encoding="utf-8"))
day, data = next(iter(archive["t17"].items()))
trial = data["trials"][0]
print(trial["ref"], "→", trial["hyp"])
print(f"{day}: trial word errors {trial['we']}/{trial['nw']}")
for participant in ("t17", "t18"):
    result = build_jude26a(participant)
    print(f"{participant}: {len(result['sessions'])} sessions, {sum(s['n'] for s in result['sessions'])} eligible trials; pooled {result['pooled']}")

how very right

 → how very right
t17.2024.04.25: trial word errors 0/3
t17: 8 sessions, 134 eligible trials; pooled {'cer': 21.52, 'wer': 14.65}
t18: 7 sessions, 140 eligible trials; pooled {'cer': 8.41, 'wer': 7.63}


## Reproduce and score

[Common reader guide](../../docs/20261124_repository_migration/reader_workflow.md) · [Study evaluation protocol](docs/20261124_evaluation_protocol/README.md)

### Train and infer

To check a single session, use the NPZ made above, then predict its held-out trials.

```powershell
$study = "studies/20260316_jude"
$date = Get-Date -Format yyyyMMdd
$run = "$study/results/${date}_dryad_cz8w9gjjk__npz__ctc_greedy__single_session_reader/runtime"
$inputFile = "$study/data/derived/dryad_cz8w9gjjk/npz/typing_20ms/t17/t17.2024.04.25/m1_torch_data.npz"
python "$study/code/pytorch_port/train.py" --npz "$inputFile" --out "$run" --n-features 768 --batches 1000
python "$study/code/infer.py" "$run" "$inputFile"
```

For joint training, prepare all eight sessions of the chosen participant using the block and exclusion settings in the [conversion manifest](data/splits/dryad_cz8w9gjjk__typing_conversion.csv). These PowerShell commands prepare both participants and run T17. Repeat training with `--participant t18` to include T18.

```powershell
Import-Csv "$study/data/splits/dryad_cz8w9gjjk__typing_conversion.csv" | ForEach-Object {
    python "$study/code/prepare_data.py" $_.session $_.blocks $_.test_trials $_.trial_removals
    if ($LASTEXITCODE -ne 0) { throw "TFRecord conversion failed for $($_.session)" }
    python "$study/code/pytorch_port/export_tfrecord_to_npz.py" $_.session
    if ($LASTEXITCODE -ne 0) { throw "NPZ export failed for $($_.session)" }
}
python "$study/code/train.py" --participant t17
```

### Calculate metrics

For the single session, read CER in `$run/final_report.json`. For joint training, read the participant's `multisession_summary.json` under `results/<dated-run>/runtime/runs_multisess/`. It reports raw CER and training exports the session-level logits.

For post-LM CER and WER, follow the [typing decoder setup in WSL](../../docs/20261124_repository_migration/reader_workflow.md#typing-word-decoding) and pass the joint training folder to `decode.py --input-dir`. The decoder records edit counts and rates in its own `summary.json`. `evaluate.py` recomputes online typing speed and errors using the released online blocks.
